# 3.7 — RAG Agent

In Week 2 we built a fixed RAG pipeline — every question went through retrieval.

A **RAG Agent** is smarter:
- It decides *when* to retrieve (not every question needs it)
- It can call retrieval multiple times with different queries
- It can combine retrieved knowledge with other tools

```
Question: "What is 2 + 2?"
  Agent thinks: This is math, no need to search the document.
  → calls calculator tool

Question: "What does the Gita say about duty?"
  Agent thinks: I need to look this up in the document.
  → calls rag_search tool
```

In [ ]:
!pip install langchain langchain-ollama langchain-community chromadb pypdf --quiet

## Step 1 — Build the Knowledge Base

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

# Sample knowledge base — teachings from the Bhagavad Gita
gita_passages = [
    Document(page_content="You have a right to perform your prescribed duties, but you are not entitled to the fruits of your actions. Never consider yourself the cause of the results of your activities, and never be attached to not doing your duty.", metadata={'chapter': 2, 'verse': 47}),
    Document(page_content="The soul is never born nor dies at any time. It has not come into being, does not come into being, and will not come into being. It is unborn, eternal, ever-existing, and primeval. It is not slain when the body is slain.", metadata={'chapter': 2, 'verse': 20}),
    Document(page_content="One who has controlled the mind is tranquil in heat and cold, in pleasure and pain, in honor and dishonor. Such a person is always steady.", metadata={'chapter': 6, 'verse': 7}),
    Document(page_content="Let a man lift himself by his own self alone, let him not lower himself; for this self alone is the friend of oneself, and this self alone is the enemy of oneself.", metadata={'chapter': 6, 'verse': 5}),
    Document(page_content="Perform all thy actions with mind concentrated on the Divine, renouncing attachment and looking upon success and failure with an equal eye. Spirituality implies equanimity.", metadata={'chapter': 2, 'verse': 48}),
    Document(page_content="The three gates to self-destructive hell are lust, anger, and greed. Every sane man should give these up, for they lead to the degradation of the soul.", metadata={'chapter': 16, 'verse': 21}),
    Document(page_content="A person in the divine consciousness, although engaged in seeing, hearing, touching, smelling, eating, moving about, sleeping and breathing, always knows within himself that he actually does nothing at all.", metadata={'chapter': 5, 'verse': 8}),
    Document(page_content="Out of compassion for them, I, dwelling in their hearts, destroy with the shining lamp of knowledge the darkness born of ignorance.", metadata={'chapter': 10, 'verse': 11}),
    Document(page_content="Fix your mind on Me, be devoted to Me, worship Me, bow down to Me. So shall you come to Me. I promise you truly, for you are dear to Me.", metadata={'chapter': 18, 'verse': 65}),
    Document(page_content="Whatever you do, whatever you eat, whatever you offer or give away, and whatever austerities you perform — do that as an offering to Me.", metadata={'chapter': 9, 'verse': 27}),
]

embeddings = OllamaEmbeddings(model='llama3.1')
vectorstore = Chroma.from_documents(gita_passages, embedding=embeddings, collection_name='gita')
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

print(f'Knowledge base ready with {len(gita_passages)} passages.')

## Step 2 — Define the RAG Tool + Supporting Tools

In [ ]:
from langchain_core.tools import tool
import math

@tool
def search_gita(query: str) -> str:
    """Searches the Bhagavad Gita for passages relevant to a topic or question.
    Use this for any question about the Gita's teachings, philosophy, duty, soul, or Krishna.
    """
    docs = retriever.invoke(query)
    if not docs:
        return 'No relevant passages found.'
    results = []
    for doc in docs:
        ch = doc.metadata.get('chapter', '?')
        vs = doc.metadata.get('verse', '?')
        results.append(f'[Chapter {ch}, Verse {vs}]: {doc.page_content}')
    return '\n\n'.join(results)

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    try:
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        return str(eval(expression, {'__builtins__': {}}, allowed))
    except Exception as e:
        return f'Error: {e}'

@tool
def get_current_date() -> str:
    """Returns today's date."""
    from datetime import date
    return f"Today's date is {date.today().strftime('%B %d, %Y')}."

# Test the RAG tool
print('Testing search_gita:')
print(search_gita.invoke({'query': 'What does the Gita say about the soul?'})[:300])

## Step 3 — Build the RAG Agent

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

llm = ChatOllama(model='llama3.1', temperature=0)

tools = [search_gita, calculator, get_current_date]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are a knowledgeable assistant with access to the Bhagavad Gita.

Available tools:
- search_gita: Use for any question about the Gita's teachings, philosophy, or spirituality
- calculator: Use for any mathematical calculations
- get_current_date: Use when the user asks about today's date or time

For Gita questions, ALWAYS cite the chapter and verse from the search results.
For general knowledge questions not requiring tools, answer directly."""

def rag_agent(question: str) -> str:
    print(f'Q: {question}')
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question)
    ]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            print(f'A: {response.content}')
            print()
            return response.content

        for tc in response.tool_calls:
            result = tool_map[tc['name']].invoke(tc['args'])
            print(f'  [tool: {tc["name"]}]')
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

print('RAG Agent ready.')

## Step 4 — Test the Agent

Watch how the agent picks `search_gita` for spiritual questions but other tools for different questions.

In [ ]:
# Should use search_gita
rag_agent('What does the Bhagavad Gita say about duty and action?')

In [ ]:
# Should use search_gita
rag_agent('What is the Gita\'s teaching about the soul and death?')

In [ ]:
# Should use calculator — NOT search_gita
rag_agent('What is 108 multiplied by 18?')

In [ ]:
# Should answer directly — no tool needed
rag_agent('What is the Bhagavad Gita?')

In [ ]:
# Multi-step: search Gita + calculate something from the results
rag_agent('How many chapters and verses are mentioned in the passages about duty and action? Count the unique chapters.')

## Summary

| Fixed RAG Pipeline | RAG Agent |
|--------------------|----------|
| Always retrieves | Retrieves only when needed |
| Single retrieval per query | Can retrieve multiple times |
| Cannot use other tools | Combines RAG with any tool |
| Simpler to build | More flexible and powerful |

**Rule of thumb:** use a fixed pipeline for focused Q&A, use a RAG agent when your app also needs to perform actions or answer diverse question types.